# EfficientNetV2 + Grad-CAM for Oral Histopathology Classification

| Scenario | Time |
|---------|------:|
| Real (our run) | **2h40** |
| Optimistic | 1h48 |
| Average | 4h06 |
| Pessimistic | 10h12 |

> Estimated on a Kaggle T4 x2 GPU. Early stopping (patience=10) triggered around epoch 18–20 on average, given the small training sets (~150 images per dataset).

## 1. Environment & Dependencies

Input > Add Input > 
https://www.kaggle.com/datasets/sthephannysantos/oral-dataset

<br>

This section sets up the execution environment on Kaggle.
It verifies that the dataset is correctly mounted at `/kaggle/input`,
installs `timm` (pretrained model library) and `thop` (FLOPs counter),
and confirms GPU availability for accelerated training.

In [ ]:
# Montar os dados:
import os
print('Kaggle input:', os.listdir('/kaggle/input'))

In [ ]:
# Instalar dependências extras não disponíveis no ambiente padrão do Kaggle
import subprocess
subprocess.run(['pip', 'install', 'timm', 'thop', '-q'], check=True)

In [ ]:
# Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memória GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Imports
Imports all required libraries: PyTorch for deep learning, `timm` for
pretrained EfficientNetV2 models, `sklearn` for evaluation metrics,
and visualization tools (`matplotlib`, `PIL`).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.cuda.amp import autocast, GradScaler
from torch import optim
import timm
import os
import copy
import json
import time
import random

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from thop import profile

print('Todas as importações OK.')

## 3. Project Settings

Defines all global settings for the experiment:
- Dataset and output paths under `/kaggle/working/`
- Hyperparameters: 50 epochs, batch size 32, learning rate 1e-4, patience 10
- Experiment grid: 2 architectures × 3 training modes × 2 augmentation settings × 3 seeds = **72 experiments**
- Creates output folders for checkpoints, histories, confusion matrices, learning curves, GradCAM and plots.

In [ ]:
import torch, os

# ── Caminhos Kaggle ──────────────────────────────────────────────────────────
# Ajuste 'oral-dataset' para o slug do seu dataset no Kaggle se for diferente
BASE_INPUT  = '/kaggle/input/datasets/sthephannysantos/oral-dataset'
DATASET_DIR = f'{BASE_INPUT}/datasets'
RESULTS_DIR = '/kaggle/working'
# ─────────────────────────────────────────────────────────────────────────────

definitions = {
    'datasets_folder': DATASET_DIR + '/',
    'imgs1_folder':    DATASET_DIR + '/Original ROI images/',
    'imgs2_folder':    DATASET_DIR + '/images/',
    'epochs':          50,
    'batch_size':      32,
    'patience':        10,    # early stopping
    'lr':              1e-4,
    'num_workers':     2,
}

models_list      = ['tf_efficientnetv2_b0.in1k', 'tf_efficientnetv2_b1.in1k']
training_modes   = ['scratch', 'feature_extraction', 'fine_tuning']
augmentation_modes = [True, False]
seeds            = [42, 123, 2025]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

# Criar pastas de saída em /kaggle/working
for subdir in ['checkpoints', 'histories', 'confusion_matrices',
               'learning_curves', 'gradcam', 'graficos']:
    os.makedirs(f'{RESULTS_DIR}/{subdir}', exist_ok=True)

print(f'Pasta de resultados: {RESULTS_DIR}')
print(f'Conteúdo do dataset: {os.listdir(DATASET_DIR)}')

## 4. Transforms & Dataset

Defines image preprocessing pipelines for training and validation/test:
- **Train**: resize → random crop → horizontal flip → random 90° rotation → normalize (ImageNet stats)
- **Val/Test**: resize → center crop → normalize (deterministic)

`OralCancerDataset` is a custom PyTorch `Dataset` that reads images from
disk using paths and labels from the manifest CSV files.

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize(256),           # reduz mantendo proporção original
    transforms.RandomCrop(224),       # corte aleatório (augmentation)
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomChoice([
        transforms.RandomRotation(degrees=(0,   0)),
        transforms.RandomRotation(degrees=(90,  90)),
        transforms.RandomRotation(degrees=(180, 180)),
        transforms.RandomRotation(degrees=(270, 270)),
    ]),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize(256),           # reduz mantendo proporção original
    transforms.CenterCrop(224),       # corte central determinístico
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

def get_transform_train(augmentation=True):
    return transform_train if augmentation else transform_val


class OralCancerDataset(Dataset):
    def __init__(self, df, images_folder, transform=None):
        self.df            = df
        self.images_folder = images_folder
        self.transform     = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        img   = Image.open(os.path.join(self.images_folder, row['path'])).convert('RGB')
        label = int(row['label_number'])
        if self.transform:
            img = self.transform(img)
        return img, label

print('Dataset e transformações prontos.')

## 5. Utility Functions

Helper functions used throughout the pipeline:
- `set_seed`: ensures reproducibility across PyTorch, NumPy and Python random
- `create_model`: builds EfficientNetV2 in one of three modes — scratch (random weights), feature extraction (frozen backbone), or fine-tuning (all layers trainable)
- `get_model_complexity`: counts parameters and computes GFLOPs for a 224×224 input
- `denormalize`: reverses ImageNet normalization for visualization

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def create_model(architecture, mode, num_classes=3):
    if mode == 'scratch':
        model = timm.create_model(architecture, pretrained=False, num_classes=num_classes)
    elif mode == 'feature_extraction':
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)
        for p in model.parameters():
            p.requires_grad = False
        for p in model.get_classifier().parameters():
            p.requires_grad = True
    elif mode == 'fine_tuning':
        model = timm.create_model(architecture, pretrained=True, num_classes=num_classes)
    return model


def get_model_complexity(model, device):
    """Retorna (num_params, GFLOPs) para entrada 224×224."""
    num_params = sum(p.numel() for p in model.parameters())
    dummy      = torch.zeros(1, 3, 224, 224).to(device)
    model.eval()
    flops, _   = profile(model, inputs=(dummy,), verbose=False)
    return num_params, flops / 1e9


def denormalize(tensor):
    """Desfaz normalização ImageNet para visualização."""
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    img  = tensor.cpu() * std + mean
    return np.clip(img.permute(1, 2, 0).numpy(), 0, 1)


print('Utilitários prontos.')

## 6. Training & Evaluation Functions

Core training and evaluation logic:
- `train_one_epoch`: runs one full pass over the training set using mixed precision (AMP) for memory efficiency
- `validate`: evaluates the model on validation data without gradient computation
- `evaluate_test`: runs inference on the test set and returns accuracy, F1 macro, F1 weighted, confusion matrix and raw predictions

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        with autocast():                          # mixed precision
            outputs = model(images)
            loss    = criterion(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        correct    += (outputs.argmax(dim=1) == labels).sum().item()
        total      += labels.size(0)
    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs        = model(images)
            loss           = criterion(outputs, labels)
            total_loss    += loss.item()
            correct       += (outputs.argmax(dim=1) == labels).sum().item()
            total         += labels.size(0)
    return total_loss / len(loader), correct / total


def evaluate_test(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            preds  = model(images).argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())
    return {
        'acc_test':         accuracy_score(all_labels, all_preds),
        'f1_macro_test':    f1_score(all_labels, all_preds, average='macro'),
        'f1_weighted_test': f1_score(all_labels, all_preds, average='weighted'),
        'confusion_matrix': confusion_matrix(all_labels, all_preds).tolist(),
        'all_preds':        all_preds,
        'all_labels':       all_labels,
    }


print('Funções de treinamento prontas.')

## 7. Visualization Functions

Plotting utilities for result analysis:
- `plot_learning_curves`: saves loss and accuracy curves per epoch
- `plot_confusion_matrix`: saves confusion matrix for each experiment
- `generate_summary_plots`: generates comparative boxplots of F1 Weighted by architecture, training mode and augmentation, plus bar charts of model complexity (parameters and GFLOPs)

In [ ]:
def plot_learning_curves(history, name, save_dir):
    """Plota e salva curvas de loss e accuracy por época."""
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train')
    axes[0].plot(epochs, history['val_loss'],   label='Val')
    axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
    axes[0].set_title('Curva de Loss'); axes[0].legend()

    axes[1].plot(epochs, history['train_acc'], label='Train')
    axes[1].plot(epochs, history['val_acc'],   label='Val')
    axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia')
    axes[1].set_title('Curva de Acurácia'); axes[1].legend()

    fig.suptitle(name.replace('_', ' '), fontsize=10)
    plt.tight_layout()
    plt.savefig(f'{save_dir}/learning_curves/{name}.png', dpi=100, bbox_inches='tight')
    plt.close()


def plot_confusion_matrix(cm_list, num_classes, class_names, name, save_dir):
    """Plota e salva matriz de confusão."""
    cm   = np.array(cm_list)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=class_names[:num_classes])
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name.replace('_', ' '))
    plt.tight_layout()
    plt.savefig(f'{save_dir}/confusion_matrices/{name}.png', dpi=100, bbox_inches='tight')
    plt.close()


def generate_summary_plots(df, save_dir):
    """Gráficos de barras de F1, parâmetros e GFLOPs."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # F1 por arquitetura
    df.boxplot(column='f1_weighted_test', by='model', ax=axes[0])
    axes[0].set_title('F1 Weighted por Arquitetura')
    axes[0].set_xlabel('Arquitetura'); axes[0].set_ylabel('F1 Weighted')

    # F1 por modo de treinamento
    df.boxplot(column='f1_weighted_test', by='training_mode', ax=axes[1])
    axes[1].set_title('F1 Weighted por Modo')
    axes[1].set_xlabel('Modo'); axes[1].set_ylabel('')
    axes[1].tick_params(axis='x', rotation=15)

    # F1 por augmentation
    df.boxplot(column='f1_weighted_test', by='augmentation', ax=axes[2])
    axes[2].set_title('F1 Weighted por Augmentation')
    axes[2].set_xlabel('Augmentation'); axes[2].set_ylabel('')

    fig.suptitle('')
    plt.tight_layout()
    plt.savefig(f'{save_dir}/graficos/f1_comparativos.png', dpi=100, bbox_inches='tight')
    plt.show()

    # Parâmetros e GFLOPs por arquitetura
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    arch_stats = df.groupby('model')[['num_params', 'gflops']].first()

    arch_stats['num_params'].plot(kind='bar', ax=axes[0], color='steelblue', rot=15)
    axes[0].set_title('Número de Parâmetros')
    axes[0].set_ylabel('Parâmetros')
    axes[0].yaxis.set_major_formatter(
        plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

    arch_stats['gflops'].plot(kind='bar', ax=axes[1], color='darkorange', rot=15)
    axes[1].set_title('Complexidade Computacional (GFLOPs)')
    axes[1].set_ylabel('GFLOPs')

    plt.tight_layout()
    plt.savefig(f'{save_dir}/graficos/complexidade_modelos.png', dpi=100, bbox_inches='tight')
    plt.show()


print('Funções de visualização prontas.')

## 8. Grad-CAM

Implementation of Gradient-weighted Class Activation Mapping (Grad-CAM)
for model interpretability:
- `GradCAM`: uses forward/backward hooks to capture activations and gradients from the target convolutional layer, generating a heatmap highlighting regions most relevant to the predicted class
- `get_efficientnet_target_layer`: automatically identifies the last convolutional layer of EfficientNetV2 (timm)
- `select_gradcam_samples`: selects 2 samples per class from the test set — prioritizing 1 correct + 1 incorrect prediction
- `visualize_gradcam`: overlays the heatmap on the original image; shows 2 columns for correct predictions and 3 columns for incorrect ones (predicted class + true class)

In [ ]:
class GradCAM:
    """
    Usa hooks para capturar ativações e gradientes de uma camada alvo.
    """
    def __init__(self, model, target_layer):
        self.model        = model
        self.target_layer = target_layer
        self.activations  = None
        self.gradients    = None
        self._fwd = target_layer.register_forward_hook(self._save_activations)
        self._bwd = target_layer.register_full_backward_hook(self._save_gradients)

    def _save_activations(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradients(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        # Grad-CAM requer gradientes: mantemos o modelo em eval mas
        # garantimos que os hooks capturem os gradientes corretamente.
        self.model.zero_grad(set_to_none=True)
        logits = self.model(input_tensor)

        if class_idx is None:
            class_idx = logits.argmax(dim=1).item()

        logits[:, class_idx].sum().backward()

        if self.activations is None or self.gradients is None:
            raise RuntimeError(
                'Ativações/gradientes não capturados. '
                'Verifique get_efficientnet_target_layer().')

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # média global
        cam     = F.relu((weights * self.activations).sum(dim=1, keepdim=True))
        cam     = F.interpolate(cam, size=input_tensor.shape[-2:],
                                mode='bilinear', align_corners=False)
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        prob = torch.softmax(logits, dim=1)[0, class_idx].item()
        return cam.squeeze().cpu(), class_idx, prob

    def remove_hooks(self):
        self._fwd.remove()
        self._bwd.remove()


def get_efficientnet_target_layer(model):
    """
    Identifica a última camada convolucional do EfficientNetV2 (timm).
    Estratégia: conv_head → último bloco → último módulo não-linear.
    """
    # 1ª escolha: conv_head (presente na maioria dos EfficientNet do timm)
    if hasattr(model, 'conv_head'):
        return model.conv_head

    # 2ª escolha: último bloco sequential
    if hasattr(model, 'blocks') and len(model.blocks) > 0:
        last_block = model.blocks[-1]
        if isinstance(last_block, nn.Sequential):
            for m in reversed(list(last_block.children())):
                if isinstance(m, nn.Conv2d):
                    return m
        return last_block

    # Fallback genérico
    for m in reversed(list(model.children())):
        if not isinstance(m, (nn.Linear, nn.AdaptiveAvgPool2d,
                               nn.Flatten, nn.Identity)):
            return m

    raise ValueError('Camada alvo não encontrada. Inspecione model.named_modules().')


def select_gradcam_samples(model, dataloader, device, n_per_class=2):
    """
    Seleciona exatamente n_per_class amostras por classe do conjunto de teste.

    Regra de seleção (em ordem de prioridade):
      1. 1 amostra CORRETA  + 1 amostra INCORRETA  (quando ambas existem)
      2. 2 amostras CORRETAS  (quando não há erros para essa classe)
      3. 2 amostras INCORRETAS (quando não há acertos para essa classe)

    Garante que tensores são armazenados como lista Python (evita problema
    de serialização do pandas com tensores PyTorch).
    """
    model.eval()

    # Coleta todas as predições do conjunto de teste
    images_list, labels_list, preds_list = [], [], []
    with torch.no_grad():
        for images, labels in dataloader:
            batch_preds = model(images.to(device)).argmax(dim=1).cpu()
            for i in range(len(labels)):
                images_list.append(images[i])        # tensor CPU
                labels_list.append(labels[i].item())
                preds_list.append(batch_preds[i].item())

    # Índices por classe, separados em corretos e incorretos
    classes = sorted(set(labels_list))
    selected = []

    for cls in classes:
        idx_correct   = [i for i in range(len(labels_list))
                         if labels_list[i] == cls and preds_list[i] == cls]
        idx_incorrect = [i for i in range(len(labels_list))
                         if labels_list[i] == cls and preds_list[i] != cls]

        # Embaralhar com semente fixa para reprodutibilidade
        rng = np.random.default_rng(42)
        rng.shuffle(idx_correct)
        rng.shuffle(idx_incorrect)

        # Aplicar regra de seleção
        if len(idx_correct) >= 1 and len(idx_incorrect) >= 1:
            chosen_idx = [idx_correct[0], idx_incorrect[0]]
            print(f'  Classe {cls}: 1 correta + 1 incorreta')
        elif len(idx_correct) >= n_per_class:
            chosen_idx = idx_correct[:n_per_class]
            print(f'  ⚠️  Classe {cls}: sem erros — usando {n_per_class} corretas.')
        elif len(idx_incorrect) >= n_per_class:
            chosen_idx = idx_incorrect[:n_per_class]
            print(f'  ⚠️  Classe {cls}: sem acertos — usando {n_per_class} incorretas.')
        else:
            # Combinar o que existe para completar n_per_class
            chosen_idx = (idx_correct + idx_incorrect)[:n_per_class]
            n_got = len(chosen_idx)
            print(f'  ⚠️  Classe {cls}: apenas {n_got} amostras disponíveis (esperado {n_per_class}).')

        for i in chosen_idx:
            selected.append({
                'image':   images_list[i],
                'label':   labels_list[i],
                'pred':    preds_list[i],
                'correct': labels_list[i] == preds_list[i],
            })

    return selected


def visualize_gradcam(model, target_layer, sample, class_names, idx, save_dir):
    """
    Gera e salva a visualização Grad-CAM de uma amostra.

    Layout:
      - Classificação CORRETA  → 2 colunas: original | Grad-CAM (classe prevista)
      - Classificação INCORRETA → 3 colunas: original | Grad-CAM (prevista) | Grad-CAM (verdadeira)
    """
    img_tensor      = sample['image'].unsqueeze(0).to(device)
    true_label      = sample['label']
    predicted_label = sample['pred']
    is_correct      = sample['correct']

    img_display = denormalize(sample['image'])

    def _run_gradcam(class_idx):
        # Grad-CAM precisa de gradientes; garantir que não estamos bloqueados
        gc  = GradCAM(model, target_layer)
        with torch.enable_grad():
            cam, _, prob = gc.generate(img_tensor, class_idx=class_idx)
        gc.remove_hooks()
        return cam, prob

    cam_pred, prob_pred = _run_gradcam(predicted_label)

    n_cols = 3 if not is_correct else 2
    fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))

    def _plot(ax, cam, title):
        ax.imshow(img_display)
        ax.imshow(cam.numpy(), cmap='jet', alpha=0.45)
        ax.axis('off')
        ax.set_title(title, fontsize=9)

    true_name = class_names[true_label] if class_names else str(true_label)
    pred_name = class_names[predicted_label] if class_names else str(predicted_label)

    # Coluna 0: imagem original
    axes[0].imshow(img_display)
    axes[0].axis('off')
    status = '✓ Correto' if is_correct else '✗ Incorreto'
    axes[0].set_title(f'Original\nVerdadeiro: {true_name}\n{status}', fontsize=9)

    # Coluna 1: Grad-CAM para classe prevista
    _plot(axes[1], cam_pred,
          f'Grad-CAM (Previsto)\n{pred_name} ({prob_pred:.2%})')

    # Coluna 2 (só erros): Grad-CAM para classe verdadeira
    if not is_correct:
        cam_true, prob_true = _run_gradcam(true_label)
        _plot(axes[2], cam_true,
              f'Grad-CAM (Verdadeiro)\n{true_name} ({prob_true:.2%})')

    plt.tight_layout()
    status_str = 'correct' if is_correct else 'incorrect'
    fname = f'sample{idx:02d}_cls{true_label}_{status_str}_pred{predicted_label}.png'
    plt.savefig(f'{save_dir}/{fname}', dpi=100, bbox_inches='tight')
    plt.show()
    plt.close('all')
    import gc; gc.collect()  # libera memória GPU


print('Grad-CAM pronto.')


## 9. Main Training Loop

Main experiment loop iterating over all 72 combinations:
2 datasets × 2 architectures × 3 training modes × 2 augmentation settings × 3 seeds.

For each experiment:
1. Sets random seed for reproducibility
2. Builds the model and measures complexity (params + GFLOPs)
3. Trains with early stopping (patience=10) and saves the best weights
4. Evaluates on the test set and saves history, confusion matrix and learning curves
5. Appends results to a CSV incrementally — protecting against disconnection mid-run

In [ ]:
# Carregar manifests
df_epithelium = pd.read_csv(definitions['datasets_folder'] + 'manifest_split_oralepitheliumdb.csv')
df_cancer     = pd.read_csv(definitions['datasets_folder'] + 'manifest_split_multiclass_NDB-UFES.csv')

datasets_config = [
    {
        'name':        'OralEpitheliumDB',
        'train':       df_epithelium[df_epithelium['sets'] == 'train'].reset_index(drop=True),
        'val':         df_epithelium[df_epithelium['sets'] == 'val'].reset_index(drop=True),
        'test':        df_epithelium[df_epithelium['sets'] == 'test'].reset_index(drop=True),
        'folder':      definitions['imgs1_folder'],
        'num_classes': 2,
        'class_names': ['Normal', 'Dysplasia'],
    },
    {
        'name':        'NDB-UFES',
        'train':       df_cancer[df_cancer['sets'] == 'train'].reset_index(drop=True),
        'val':         df_cancer[df_cancer['sets'] == 'val'].reset_index(drop=True),
        'test':        df_cancer[df_cancer['sets'] == 'test'].reset_index(drop=True),
        'folder':      definitions['imgs2_folder'],
        'num_classes': 3,
        'class_names': ['Normal', 'Low-grade', 'High-grade'],
    },
]

print('Datasets carregados:')
for ds in datasets_config:
    total = len(ds['train']) + len(ds['val']) + len(ds['test'])
    print(f"  {ds['name']}: {total} imagens "
          f"(train={len(ds['train'])}, val={len(ds['val'])}, test={len(ds['test'])})")

In [ ]:
# ─── LOOP PRINCIPAL DE 72 EXPERIMENTOS ───────────────────────────────────────
# 2 datasets × 2 arquiteturas × 3 modos × 2 augmentation × 3 seeds = 72

all_results  = []
total_exp    = (len(datasets_config) * len(models_list) *
                len(training_modes)  * len(augmentation_modes) * len(seeds))
exp_idx      = 0
t_inicio     = time.time()

for ds in datasets_config:
    for architecture in models_list:
        for training_mode in training_modes:
            for augmentation in augmentation_modes:
                for seed in seeds:
                    exp_idx += 1
                    tag = (f"{ds['name']}_{architecture}_{training_mode}"
                           f"_aug{augmentation}_seed{seed}")
                    print(f'\n[{exp_idx}/{total_exp}] {tag}')

                    set_seed(seed)

                    # — DataLoaders ——————————————————————————————————————————
                    nw = definitions['num_workers']
                    pin = torch.cuda.is_available()

                    train_loader = DataLoader(
                        OralCancerDataset(ds['train'], ds['folder'],
                                          get_transform_train(augmentation)),
                        batch_size=definitions['batch_size'],
                        shuffle=True, num_workers=nw, pin_memory=pin)

                    val_loader = DataLoader(
                        OralCancerDataset(ds['val'], ds['folder'], transform_val),
                        batch_size=definitions['batch_size'],
                        shuffle=False, num_workers=nw, pin_memory=pin)

                    test_loader = DataLoader(
                        OralCancerDataset(ds['test'], ds['folder'], transform_val),
                        batch_size=definitions['batch_size'],
                        shuffle=False, num_workers=nw, pin_memory=pin)

                    # — Modelo ————————————————————————————————————————————————
                    model     = create_model(architecture, training_mode,
                                             ds['num_classes']).to(device)
                    criterion = nn.CrossEntropyLoss()
                    optimizer = optim.Adam(model.parameters(),
                                          lr=definitions['lr'])
                    scaler    = GradScaler()

                    num_params, gflops = get_model_complexity(model, device)

                    # — Treinamento ———————————————————————————————————————————
                    best_val_acc = -1
                    best_wts     = copy.deepcopy(model.state_dict())
                    best_epoch   = 0
                    wait         = 0
                    history      = {'train_loss': [], 'val_loss': [],
                                    'train_acc':  [], 'val_acc':  []}

                    for epoch in range(definitions['epochs']):
                        tl, ta = train_one_epoch(model, train_loader,
                                                 criterion, optimizer,
                                                 scaler, device)
                        vl, va = validate(model, val_loader, criterion, device)

                        history['train_loss'].append(tl)
                        history['val_loss'].append(vl)
                        history['train_acc'].append(ta)
                        history['val_acc'].append(va)

                        print(f'  Ep {epoch+1:02d} | '
                              f'train_loss={tl:.4f} train_acc={ta:.4f} | '
                              f'val_loss={vl:.4f} val_acc={va:.4f}')

                        if va > best_val_acc:
                            best_val_acc = va
                            best_epoch   = epoch + 1
                            best_wts     = copy.deepcopy(model.state_dict())
                            wait         = 0
                        else:
                            wait += 1
                            if wait >= definitions['patience']:
                                print(f'  Early stop na época {epoch+1}')
                                break

                    model.load_state_dict(best_wts)

                    # — Avaliação no teste ————————————————————————————————————
                    test_metrics = evaluate_test(model, test_loader, device)
                    print(f"  TEST acc={test_metrics['acc_test']:.4f} "
                          f"f1_macro={test_metrics['f1_macro_test']:.4f} "
                          f"f1_weighted={test_metrics['f1_weighted_test']:.4f}")

                    # — Salvar checkpoint ————————————————————————————————————
                    ckpt_path = f"{RESULTS_DIR}/checkpoints/{tag}.pth"
                    torch.save(best_wts, ckpt_path)

                    # — Salvar histórico e matriz de confusão ─────────────────
                    with open(f'{RESULTS_DIR}/histories/{tag}.json', 'w') as f:
                        json.dump(history, f)
                    with open(f'{RESULTS_DIR}/confusion_matrices/{tag}.json', 'w') as f:
                        json.dump(test_metrics['confusion_matrix'], f)

                    # — Gráficos: curvas e matriz ——————————————————————————————
                    plot_learning_curves(history, tag, RESULTS_DIR)
                    plot_confusion_matrix(
                        test_metrics['confusion_matrix'],
                        ds['num_classes'], ds['class_names'],
                        tag, RESULTS_DIR)

                    # — Registro de resultado ————————————————————————————————
                    record = {
                        'repetition':       seeds.index(seed) + 1,
                        'seed':             seed,
                        'dataset':          ds['name'],
                        'model':            architecture,
                        'training_mode':    training_mode,
                        'augmentation':     augmentation,
                        'acc_test':         test_metrics['acc_test'],
                        'f1_macro_test':    test_metrics['f1_macro_test'],
                        'f1_weighted_test': test_metrics['f1_weighted_test'],
                        'num_params':       num_params,
                        'gflops':           gflops,
                        'best_epoch':       best_epoch,
                        'val_acc_best':     best_val_acc,
                        'model_path':       ckpt_path,
                    }
                    all_results.append(record)

                    # — Salvamento incremental (protege contra desconexão) ────
                    csv_path = f'{RESULTS_DIR}/all_experiments_results.csv'
                    pd.DataFrame(all_results).to_csv(csv_path, index=False)
                    print(f'  💾 CSV atualizado ({exp_idx}/{total_exp} experimentos)')

# — Finalização ───────────────────────────────────────────────────────────────
df_results = pd.DataFrame(all_results)
df_results.to_csv(csv_path, index=False)
print(f'\n✅ Todos os experimentos concluídos em '
      f'{(time.time()-t_inicio)/60:.1f} min')
print(f'CSV salvo em: {csv_path}')

## 10. Results Analysis: Mean & Std

Loads the results CSV and computes mean ± standard deviation across the 3 seeds
for each combination of dataset, model, training mode and augmentation.
Saves a summary table (`summary_mean_std.csv`) for reporting.

In [ ]:
# Carregar CSV (útil se executar esta célula separadamente depois)
df_results = pd.read_csv(f'{RESULTS_DIR}/all_experiments_results.csv')

group_keys   = ['dataset', 'model', 'training_mode', 'augmentation']
metric_cols  = ['acc_test', 'f1_macro_test', 'f1_weighted_test']

summary = (
    df_results
    .groupby(group_keys)[metric_cols]
    .agg(['mean', 'std'])
    .round(4)
)
summary.columns = ['_'.join(c) for c in summary.columns]
summary = summary.reset_index()

summary_path = f'{RESULTS_DIR}/summary_mean_std.csv'
summary.to_csv(summary_path, index=False)
print(f'Tabela de média/desvio salva em: {summary_path}\n')
display(summary)

## 11. Final Comparative Plots

Generates three sets of visualizations from the final results:
1. **Comparative F1 boxplots** — by architecture, training mode and augmentation
2. **All confusion matrices** — one per experiment (72 total), with absolute counts and percentages
3. **Best confusion matrices** — best experiment per dataset, per training mode and per model, for each of the three metrics (F1 Macro, F1 Weighted, Accuracy)

In [ ]:
# ── Gráficos de resumo ────────────────────────────────────────────────────────
df_results = pd.read_csv(f'{RESULTS_DIR}/all_experiments_results.csv')
generate_summary_plots(df_results, RESULTS_DIR)
print('Gráficos salvos em:', f'{RESULTS_DIR}/graficos/')

In [ ]:
import json, glob, re
import seaborn as sns 

# ── Matrizes de confusão (todas as combinações) ───────────────────────────────
import json, glob, re

cm_dir  = f'{RESULTS_DIR}/confusion_matrices'
fig_dir = f'{RESULTS_DIR}/graficos/confusion_matrices'
os.makedirs(fig_dir, exist_ok=True)

history_files = glob.glob(f'{RESULTS_DIR}/histories/*.json')
print(f'\nGerando {len(history_files)} matrizes de confusão...')

class_names_map = {
    'OralEpitheliumDB': ['Healthy', 'Severe'],
    'NDB-UFES': ['Leukoplakia\nw/ Dysplasia', 'Leukoplakia\nw/o Dysplasia', 'OSCC']
}

for history_file in sorted(history_files):
    filename = os.path.basename(history_file)[:-5]
    match = re.match(
        r'^(OralEpitheliumDB|NDB-UFES)_(.+?)_(scratch|feature_extraction|fine_tuning)_(aug(True|False))_seed(\d+)$',
        filename
    )
    if not match:
        print(f'⚠ Ignorado: {filename}')
        continue

    dataset_name  = match.group(1)
    model_name    = match.group(2)
    training_mode = match.group(3)
    augmentation  = match.group(5) == 'True'
    seed          = int(match.group(6))

    cm_path = f'{cm_dir}/{filename}.json'
    try:
        with open(cm_path) as f:
            cm = np.array(json.load(f))

        cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
        annot  = np.array([[f'{cm[i][j]}\n({cm_pct[i][j]:.1f}%)'
                            for j in range(cm.shape[1])]
                            for i in range(cm.shape[0])])
        labels = class_names_map.get(dataset_name, [str(i) for i in range(cm.shape[0])])

        fig, ax = plt.subplots(figsize=(10, 7))
        sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                    xticklabels=labels, yticklabels=labels)
        aug_text = 'Com Aug' if augmentation else 'Sem Aug'
        ax.set_title(f'Matriz de Confusão — {dataset_name}\n{model_name} | {training_mode} | {aug_text} | Seed: {seed}')
        ax.set_ylabel('Classe Verdadeira')
        ax.set_xlabel('Classe Predita')
        plt.tight_layout()
        plt.savefig(f'{fig_dir}/{filename}.png', dpi=100, bbox_inches='tight')
        plt.close('all')
        print(f'  ✅ {filename}')

    except FileNotFoundError:
        print(f'  ✗ CM não encontrada: {cm_path}')

print('\n✅ Matrizes de confusão concluídas!')

In [ ]:
# ── Matrizes de confusão dos melhores experimentos ────────────────────────────
metrics_to_check = {
    'f1_macro_test':    'F1 Macro',
    'f1_weighted_test': 'F1 Weighted',
    'acc_test':         'Acurácia',
}

criterios = {
    'geral':         ['dataset'],
    'training_mode': ['dataset', 'training_mode'],
    'modelo':        ['dataset', 'model'],
}

fig_best_dir = f'{RESULTS_DIR}/graficos/confusion_matrices_best'
os.makedirs(fig_best_dir, exist_ok=True)

for metric_col, metric_label in metrics_to_check.items():
    for criterio_name, group_cols in criterios.items():
        # Pega a linha individual com o maior valor da métrica por grupo
        best_rows = (
            df_results
            .loc[df_results.groupby(group_cols)[metric_col].idxmax()]
            .drop_duplicates(subset=group_cols)
        )

        for _, row in best_rows.iterrows():
            dataset_name  = row['dataset']
            model_name    = row['model']
            training_mode = row['training_mode']
            augmentation  = row['augmentation']
            seed          = int(row['seed'])  # ← seed real do melhor resultado

            filename = f'{dataset_name}_{model_name}_{training_mode}_aug{augmentation}_seed{seed}'
            cm_path  = f'{cm_dir}/{filename}.json'

            try:
                with open(cm_path) as f:
                    cm = np.array(json.load(f))

                cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
                annot  = np.array([[f'{cm[i][j]}\n({cm_pct[i][j]:.1f}%)'
                                    for j in range(cm.shape[1])]
                                    for i in range(cm.shape[0])])
                labels   = class_names_map.get(dataset_name, [str(i) for i in range(cm.shape[0])])
                aug_text = 'Com Aug' if augmentation else 'Sem Aug'

                fig, ax = plt.subplots(figsize=(10, 7))
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax,
                            xticklabels=labels, yticklabels=labels)
                ax.set_title(
                    f'Melhor por {criterio_name} ({metric_label})\n'
                    f'{dataset_name} | {model_name} | {training_mode} | {aug_text} | Seed: {seed}\n'
                    f'{metric_label}: {row[metric_col]:.4f}'
                )
                ax.set_ylabel('Classe Verdadeira')
                ax.set_xlabel('Classe Predita')
                plt.tight_layout()

                fname = f'best_{criterio_name}_{metric_col}_{filename}.png'
                plt.savefig(f'{fig_best_dir}/{fname}', dpi=100, bbox_inches='tight')
                plt.show()       
                plt.close('all')
                print(f'✅ {fname}')

            except FileNotFoundError:
                print(f'✗ CM não encontrada: {cm_path}')

print('\n✅ Matrizes dos melhores experimentos concluídas!')

## 12. Grad-CAM on Best Models

Applies Grad-CAM to the best-performing model per dataset (ranked by F1 Weighted).
Loads the saved weights, identifies the target convolutional layer, selects
representative test samples and saves heatmap visualizations to `/kaggle/working/gradcam/`.

In [ ]:
df_results = pd.read_csv(f'{RESULTS_DIR}/all_experiments_results.csv')

# Mapa para configs dos datasets
ds_map = {ds['name']: ds for ds in datasets_config}

for ds_name in df_results['dataset'].unique():
    sep = '=' * 60
    print(f'\n{sep}')
    print(f'Grad-CAM — {ds_name}')
    print(sep)

    # Selecionar o melhor modelo para este dataset (maior F1-weighted)
    best_row = (
        df_results[df_results['dataset'] == ds_name]
        .sort_values('f1_weighted_test', ascending=False)
        .iloc[0]
    )
    print(f"Melhor modelo: {best_row['model']} | "
          f"{best_row['training_mode']} | "
          f"aug={best_row['augmentation']} | "
          f"seed={int(best_row['seed'])} | "
          f"F1w={best_row['f1_weighted_test']:.4f}")

    ds_cfg = ds_map[ds_name]

    # Carregar pesos do melhor modelo
    model = create_model(
        best_row['model'],
        best_row['training_mode'],
        ds_cfg['num_classes']
    )
    
    state = torch.load(
        best_row['model_path'],
        map_location=device
    )
    
    state = {
        k: v for k, v in state.items()
        if "total_ops" not in k
        and "total_params" not in k
    }
    
    msg = model.load_state_dict(state, strict=False)
    print(msg)
    
    model.to(device).eval()

    # Identificar camada convolucional alvo para o Grad-CAM
    target_layer = get_efficientnet_target_layer(model)
    print(f'Camada alvo: {target_layer.__class__.__name__}')

    # DataLoader do conjunto de teste (sem shuffle para reprodutibilidade)
    test_loader_gc = DataLoader(
        OralCancerDataset(ds_cfg['test'], ds_cfg['folder'], transform_val),
        batch_size=8, shuffle=False,
        num_workers=definitions['num_workers'])

    # Selecionar 2 amostras por classe (1 correta + 1 incorreta quando possível)
    print('Selecionando amostras para Grad-CAM:')
    samples = select_gradcam_samples(model, test_loader_gc, device, n_per_class=2)
    print(f'{len(samples)} amostras selecionadas no total ')
    print(f'  ({len(ds_cfg["class_names"])} classes × 2 amostras/classe).')

    # Gerar e salvar visualizações
    gc_dir = f'{RESULTS_DIR}/gradcam/{ds_name}'
    os.makedirs(gc_dir, exist_ok=True)

    for i, sample in enumerate(samples):
        status = 'CORRETA' if sample['correct'] else 'INCORRETA'
        cls_name = ds_cfg['class_names'][sample['label']]
        pred_name = ds_cfg['class_names'][sample['pred']]
        print(f'  [{i+1}/{len(samples)}] Classe: {cls_name} | '
              f'Predição: {pred_name} | {status}')
        visualize_gradcam(model, target_layer, sample,
                          ds_cfg['class_names'], i, gc_dir)

    print(f'Visualizações salvas em: {gc_dir}')

print('\n✅ Grad-CAM concluído!')
